In [4]:
# Python dependencies
import pandas as pd
import tensorflow as tf
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder

# Helper function for CIFAR100
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict

# Define the path to the data directory
path_to_data_dir = "../data"

In [5]:
## Load and partition datasets
def load_cifar10():
    print("Loading CIFAR-10 dataset...")
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    data_x = np.concatenate((x_train, x_test))
    data_y = np.concatenate((y_train, y_test))
    data = np.array(list(zip(data_x, data_y)), dtype=object)
    
    # Partition data into buckets according to the class
    class_buckets = {}
    for label in np.unique(data_y):
        # data_y is shape (N, 1), so flatten to (N,)
        mask = (data_y.flatten() == label)
        class_buckets[label] = data[mask]
        
    ## Print class distribution
    print(np.unique(data_y, return_counts=True))
     
    ### Perform manual split
    splits = {"0": np.concatenate((class_buckets[0][0:750],class_buckets[1][0:750], class_buckets[2])),
                "1": np.concatenate((class_buckets[0][750:1500],class_buckets[1][750:1500], class_buckets[3])),
                "2": np.concatenate((class_buckets[0][1500:2250],class_buckets[1][1500:2250], class_buckets[4])),
                "3": np.concatenate((class_buckets[0][2250:3000],class_buckets[1][2250:3000], class_buckets[5])),
                "4": np.concatenate((class_buckets[0][3000:3750],class_buckets[1][3000:3750], class_buckets[6])),
                "5": np.concatenate((class_buckets[0][3750:4500],class_buckets[1][3750:4500], class_buckets[7])),
                "6": np.concatenate((class_buckets[0][4500:5250],class_buckets[1][4500:5250], class_buckets[8])),
                "7": np.concatenate((class_buckets[0][5250:6000],class_buckets[1][5250:6000], class_buckets[9]))}
    for split in splits:
        np.random.shuffle(splits[split])
        print(f"Client {split} has {len(splits[split])} samples.")
    # Create folder
    os.makedirs(f"{path_to_data_dir}/generated/cifar10_gen/", exist_ok=True)
    for split in splits:
        np.save(f"{path_to_data_dir}/generated/cifar10_gen/client_{split}.npy", splits[split])
    return data
    
def load_cifar100():
    print("Loading CIFAR-100 dataset...")
    train_data = unpickle(f"{path_to_data_dir}/cifar100/train")
    x_train = train_data[b'data']
    y_train = np.array(train_data[b'fine_labels'])
    test_data = unpickle(f"{path_to_data_dir}/cifar100/test")
    x_test = test_data[b'data']
    y_test = np.array(test_data[b'fine_labels'])
    data_x = np.concatenate((x_train, x_test))
    data_y = np.concatenate((y_train, y_test))
    data = np.array(list(zip(data_x, data_y)), dtype=object)
    
    # Partition data into buckets according to the class
    class_buckets = {}
    for label in np.unique(data_y):
        # data_y is shape (N, 1), so flatten to (N,)
        mask = (data_y.flatten() == label)
        class_buckets[label] = data[mask]
     
    ## Print class distribution
    print(np.unique(data_y, return_counts=True))
     
    ### Perform manual split
    # Fix: class_buckets is a dict, not a list, so use list comprehension over sorted keys
    # Each bucket is class_buckets[label], where label in 0..99
    splits = {
        "0": np.concatenate([class_buckets[label][0:75] for label in range(20)] + [class_buckets[label] for label in range(20, 30)]),
        "1": np.concatenate([class_buckets[label][75:150] for label in range(20)] + [class_buckets[label] for label in range(30, 40)]),
        "2": np.concatenate([class_buckets[label][150:225] for label in range(20)] + [class_buckets[label] for label in range(40, 50)]),
        "3": np.concatenate([class_buckets[label][225:300] for label in range(20)] + [class_buckets[label] for label in range(50, 60)]),
        "4": np.concatenate([class_buckets[label][300:375] for label in range(20)] + [class_buckets[label] for label in range(60, 70)]),
        "5": np.concatenate([class_buckets[label][375:450] for label in range(20)] + [class_buckets[label] for label in range(70, 80)]),
        "6": np.concatenate([class_buckets[label][450:525] for label in range(20)] + [class_buckets[label] for label in range(80, 90)]),
        "7": np.concatenate([class_buckets[label][525:600] for label in range(20)] + [class_buckets[label] for label in range(90, 100)]),
    }

    for split in splits:
        print(f"Client {split} has {len(splits[split])} samples.")
        np.random.shuffle(splits[split])
        
          
    # Create folder
    os.makedirs(f"{path_to_data_dir}/generated/cifar100_gen", exist_ok=True)
    for split in splits:
        np.save(f"{path_to_data_dir}/generated/cifar100_gen/client_{split}.npy", splits[split])
    return data
    
def load_adult():
    print("Loading Adult dataset...")
    column_names = [
        "age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
    ]
    train_data = pd.read_csv(f"{path_to_data_dir}/adult/adult.data", names=column_names, skiprows=1)
    test_data = pd.read_csv(f"{path_to_data_dir}/adult/adult.test", names=column_names, skiprows=1)
    data = pd.concat([train_data, test_data], axis=0, ignore_index=True)
    #{' <=50K': 24719, ' <=50K.': 12434, ' >50K': 7841, ' >50K.': 3846}
    target_column = 'income'
    data[target_column] = data[target_column].apply(
        lambda x: ">50K" if str(x).strip().startswith('>') else "<=50K"
    )
    data = data.dropna()
    categorical_columns = [
            'workclass', 'education', 'marital-status', 'occupation', 'relationship',
            'race', 'sex', 'native-country'
    ]
    numerical_columns = [
            'age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week'
    ]
    
    for col in categorical_columns:
        data[col] = LabelEncoder().fit_transform(data[col])
    
    # Partition data into buckets according to the class using pandas only
    class_buckets = {label: subset.copy() for label, subset in data.groupby(target_column)}
    print(class_buckets.keys())

    
    ## Print class distribution
    class_distribution = {label: len(bucket) for label, bucket in class_buckets.items()}
    print(class_distribution)
    
    ## Perform the manual split into 8 partitions
    # Use pd.concat instead of .append for DataFrame objects
    splits = {
        "0": pd.concat([class_buckets['<=50K'].iloc[0:4640], class_buckets['>50K'].iloc[0:10000]], ignore_index=True),
        "1": pd.concat([class_buckets['<=50K'].iloc[4640:9280], class_buckets['>50K'].iloc[10000:10050]], ignore_index=True),
        "2": pd.concat([class_buckets['<=50K'].iloc[9280:13920], class_buckets['>50K'].iloc[10050:10200]], ignore_index=True),
        "3": pd.concat([class_buckets['<=50K'].iloc[13920:18560], class_buckets['>50K'].iloc[10200:10500]], ignore_index=True),
        "4": pd.concat([class_buckets['<=50K'].iloc[18560:23200], class_buckets['>50K'].iloc[10587:10687]], ignore_index=True),
        "5": pd.concat([class_buckets['<=50K'].iloc[23200:27840], class_buckets['>50K'].iloc[10687:10987]], ignore_index=True),
        "6": pd.concat([class_buckets['<=50K'].iloc[27840:32480], class_buckets['>50K'].iloc[10987:11000]], ignore_index=True),
        "7": pd.concat([class_buckets['<=50K'].iloc[32480:37120], class_buckets['>50K'].iloc[11000:11687]], ignore_index=True)
    }

    for split in splits:
        splits[split] = splits[split].sample(frac=1).reset_index(drop=True)
        print(f"Client {split} has {len(splits[split])} samples.")
        os.makedirs(f"{path_to_data_dir}/generated/adult_gen", exist_ok=True)
        splits[split].to_csv(f"{path_to_data_dir}/generated/adult_gen/client_{split}.csv", index=False)
    
    return data
    
def load_covertype():
    print("Loading Covertype dataset...")
    column_names = [
            "Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology",
            "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways",
            "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
            "Horizontal_Distance_To_Fire_Points"
        ] + [f"Wilderness_Area_{i}" for i in range(4)] + \
        [f"Soil_Type_{i}" for i in range(40)] + ["Cover_Type"]

    ## Print class distribution
    data = pd.read_csv(f"{path_to_data_dir}/covertype/covtype.data.gz", header=None, names=column_names)
    
    # Partition data into buckets according to the class using pandas only
    class_buckets = {label: subset.copy() for label, subset in data.groupby("Cover_Type")}
    print(class_buckets.keys())

    ## Print class distribution
    class_distribution = {label: len(bucket) for label, bucket in class_buckets.items()}
    print(class_distribution)
    
    ## Perform the manual split into 8 partitions
    splits = {
        "0": pd.concat([class_buckets[1].iloc[0:26480], class_buckets[2].iloc[0:35412], class_buckets[3].iloc[0:15000], class_buckets[4], class_buckets[5], class_buckets[7].iloc[0:20000]], ignore_index=True),
        "1": pd.concat([class_buckets[1].iloc[26480:52960], class_buckets[2].iloc[35412:70824], class_buckets[3].iloc[15000:16000]], ignore_index=True),
        "2": pd.concat([class_buckets[1].iloc[52960:79440], class_buckets[2].iloc[70824:106236], class_buckets[3].iloc[16000:18000], class_buckets[7].iloc[20000:20110]], ignore_index=True),
        "3": pd.concat([class_buckets[1].iloc[79440:105920], class_buckets[2].iloc[106236:141648], class_buckets[3].iloc[18000:19000], class_buckets[6]], ignore_index=True),
        "4": pd.concat([class_buckets[1].iloc[105920:132400], class_buckets[2].iloc[141648:177060], class_buckets[3].iloc[19000:22345]], ignore_index=True),
        "5": pd.concat([class_buckets[1].iloc[132400:158880], class_buckets[2].iloc[177060:212472], class_buckets[3].iloc[22345:26814]], ignore_index=True),
        "6": pd.concat([class_buckets[1].iloc[158880:185360], class_buckets[2].iloc[212472:247884], class_buckets[3].iloc[26814:31283]], ignore_index=True),
        "7": pd.concat([class_buckets[1].iloc[185360:211840], class_buckets[2].iloc[247884:283296], class_buckets[3].iloc[31283:35752], class_buckets[7].iloc[20110:20510]], ignore_index=True) 
    }
    
    for split in splits:
        splits[split] = splits[split].sample(frac=1).reset_index(drop=True)
        print(f"Client {split} has {len(splits[split])} samples.")
        os.makedirs(f"{path_to_data_dir}/generated/covertype_gen", exist_ok=True)
        splits[split].to_csv(f"{path_to_data_dir}/generated/covertype_gen/client_{split}.csv", index=False)
    
    return data

In [6]:
dataset_loading_list = [load_cifar10] #, load_cifar100,load_covertype, load_adult]

for dataset_func in dataset_loading_list:
    dataset_func()

Loading CIFAR-10 dataset...
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint8), array([6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]))
Client 0 has 7500 samples.
Client 1 has 7500 samples.
Client 2 has 7500 samples.
Client 3 has 7500 samples.
Client 4 has 7500 samples.
Client 5 has 7500 samples.
Client 6 has 7500 samples.
Client 7 has 7500 samples.
